# Flight Operations Analytics (2018–2022) - Pandas Local
Construcción de Datamart (Modelo Estrella) en Formato Parquet y ORC sin Spark - 100% Local

## 0) Instalación e Imports

In [7]:
import subprocess
import sys

# Instalar paquetes necesarios si no están disponibles
packages = ['pandas', 'pyarrow', 'numpy']

for package in packages:
    try:
        __import__(package)
        print(f"✓ {package} ya está instalado")
    except ImportError:
        print(f"Instalando {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} instalado")

print("\nImportando librerías...")
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
import os
import re
import time
from pathlib import Path

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)

✓ pandas ya está instalado
✓ pyarrow ya está instalado
✓ numpy ya está instalado

Importando librerías...
Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Pandas: 3.0.3
PyArrow: 24.0.0


## 1) Utilidades y Configuración Local

In [13]:
# ============================================================
# UTILIDADES
# ============================================================

def hr_size(num_bytes: int) -> str:
    """Convertir bytes a formato legible (B, KB, MB, GB, TB)"""
    size = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} PB"


def dir_size(path: str) -> int:
    """Calcular tamaño total de un directorio"""
    total = 0
    if not os.path.exists(path):
        return 0
    
    for root, dirs, files in os.walk(path):
        for file in files:
            file_path = os.path.join(root, file)
            if os.path.exists(file_path):
                total += os.path.getsize(file_path)
    return total


def list_files(path: str):
    """Listar archivos en un directorio"""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Ruta no existe: {path}")
    
    return [
        os.path.join(path, f)
        for f in os.listdir(path)
        if os.path.isfile(os.path.join(path, f))
    ]


def find_col(patterns, columns):
    """Buscar columna por patrón regex"""
    for p in patterns:
        rx = re.compile(p, re.IGNORECASE)
        for c in columns:
            if rx.fullmatch(c) or rx.search(c):
                return c
    return None


# ============================================================
# DEFINIR RUTAS PARA WINDOWS LOCAL
# ============================================================

SCHEMA = "flights"

# Carpeta base local (ajusta si lo necesitas)
BASE_VOLUME_PATH = "."

# Rutas intermedias
PATH_WORKLOAD = os.path.join(BASE_VOLUME_PATH, "workload")
PATH_LANDING = os.path.join(BASE_VOLUME_PATH, "landing", "flights")
PATH_CURATED = os.path.join(BASE_VOLUME_PATH, "curated", "flights")
PATH_FUNCTIONAL = os.path.join(BASE_VOLUME_PATH, "functional", "datamart")

# Input: CSV o Parquet en workload
KAGGLE_INPUT_PATH = PATH_WORKLOAD

# Crear carpetas si no existen
for path in [PATH_WORKLOAD, PATH_LANDING, PATH_CURATED, PATH_FUNCTIONAL]:
    os.makedirs(path, exist_ok=True)

print("=== CONFIGURACIÓN ===")
print("BASE_VOLUME_PATH:", BASE_VOLUME_PATH)
print("PATH_LANDING:", PATH_LANDING)
print("PATH_CURATED:", PATH_CURATED)
print("PATH_FUNCTIONAL:", PATH_FUNCTIONAL)

=== CONFIGURACIÓN ===
BASE_VOLUME_PATH: .
PATH_LANDING: .\landing\flights
PATH_CURATED: .\curated\flights
PATH_FUNCTIONAL: .\functional\datamart


## 2) LANDING — Ingestión Raw

In [9]:
# Buscar todos los CSV y Parquet en PATH_WORKLOAD
print("Buscando archivos de entrada en:", KAGGLE_INPUT_PATH)

candidates = []
if os.path.exists(KAGGLE_INPUT_PATH):
    for f in os.listdir(KAGGLE_INPUT_PATH):
        if f.lower().endswith(('.csv', '.parquet')):
            candidates.append(os.path.join(KAGGLE_INPUT_PATH, f))

if not candidates:
    print("⚠ No se encontraron .csv o .parquet")
    print("Por favor, coloca tus archivos en:", KAGGLE_INPUT_PATH)
    print("\nArchivos detectados:")
    if os.path.exists(KAGGLE_INPUT_PATH):
        for f in os.listdir(KAGGLE_INPUT_PATH)[:10]:
            print("-", f)
else:
    print(f"\n✓ {len(candidates)} archivo(s) encontrado(s):")
    for f in candidates:
        print(f"  - {os.path.basename(f)} ({hr_size(os.path.getsize(f))})")

Buscando archivos de entrada en: .\workload

✓ 5 archivo(s) encontrado(s):
  - Combined_Flights_2018.parquet (215.32 MB)
  - Combined_Flights_2019.parquet (294.37 MB)
  - Combined_Flights_2020.parquet (174.65 MB)
  - Combined_Flights_2021.parquet (231.74 MB)
  - Combined_Flights_2022.parquet (142.72 MB)


In [11]:
# Cargar y combinar TODOS los archivos candidatos
print("\nCombinando archivos...")

all_dfs = []

for file_path in candidates:
    file_name = os.path.basename(file_path)
    try:
        if file_path.lower().endswith('.csv'):
            print(f"  Leyendo CSV: {file_name}")
            df_temp = pd.read_csv(file_path)
        else:
            print(f"  Leyendo Parquet: {file_name}")
            df_temp = pd.read_parquet(file_path, engine='pyarrow')
        
        print(f"    → {len(df_temp)} filas, {len(df_temp.columns)} columnas")
        all_dfs.append(df_temp)
    except Exception as e:
        print(f"    ⚠ Error al leer {file_name}: {str(e)}")

if all_dfs:
    # Combinar todos los DataFrames
    df_raw = pd.concat(all_dfs, ignore_index=True)
    print(f"\n✓ Archivos combinados: {len(df_raw)} filas, {len(df_raw.columns)} columnas")
else:
    print("⚠ No se pudieron leer archivos")
    df_raw = None

if df_raw is not None:
    print("\nPrimeras filas:")
    print(df_raw.head(3))
    print("\nColumnas detectadas:", df_raw.columns.tolist())


Combinando archivos...
  Leyendo Parquet: Combined_Flights_2018.parquet
    → 5689512 filas, 61 columnas
  Leyendo Parquet: Combined_Flights_2019.parquet
    → 8091684 filas, 61 columnas
  Leyendo Parquet: Combined_Flights_2020.parquet
    → 5022397 filas, 61 columnas
  Leyendo Parquet: Combined_Flights_2021.parquet
    → 6311871 filas, 61 columnas
  Leyendo Parquet: Combined_Flights_2022.parquet
    → 4078318 filas, 61 columnas


MemoryError: Unable to allocate 1.96 GiB for an array with shape (9, 29193782) and data type float64

In [14]:
# Escribir en landing como Parquet
landing_path = os.path.join(PATH_LANDING, "flights_raw_parquet")
os.makedirs(landing_path, exist_ok=True)

landing_file = os.path.join(landing_path, "data.parquet")
df_raw.to_parquet(landing_file, engine='pyarrow', index=False, compression='snappy')

print(f"✓ Landing written: {landing_file}")
print(f"  Tamaño: {hr_size(os.path.getsize(landing_file))}")

NameError: name 'df_raw' is not defined

In [15]:
landing_path = os.path.join(PATH_LANDING, "flights_raw_parquet")
os.makedirs(landing_path, exist_ok=True)

landing_file = os.path.join(landing_path, "data.parquet")

## 3) CURATED — Mapeo y Limpieza

In [16]:
# Leer landing
df_land = pd.read_parquet(landing_file, engine='pyarrow')
cols = df_land.columns.tolist()

print(f"Columnas en landing: {len(cols)}")

# Mapeo canónico: canonical_name -> source_column_name
mapping = {
    # Core
    "FlightDate": "FlightDate",
    "Airline": "Airline",
    "Operating_Airline": "Operating_Airline",
    "Origin": "Origin",
    "Dest": "Dest",
    
    # Cancel/Divert
    "Cancelled": "Cancelled",
    "Diverted": "Diverted",
    "DivAirportLandings": "DivAirportLandings",
    
    # Scheduled / actual times
    "CRSDepTime": "CRSDepTime",
    "DepTime": "DepTime",
    "CRSArrTime": "CRSArrTime",
    "ArrTime": "ArrTime",
    
    # Delays
    "DepDelayMinutes": "DepDelayMinutes",
    "ArrDelayMinutes": "ArrDelayMinutes",
    "DepDel15": "DepDel15",
    "ArrDel15": "ArrDel15",
    "DepartureDelayGroups": "DepartureDelayGroups",
    "ArrivalDelayGroups": "ArrivalDelayGroups",
    
    # Durations
    "AirTime": "AirTime",
    "CRSElapsedTime": "CRSElapsedTime",
    "ActualElapsedTime": "ActualElapsedTime",
    "Distance": "Distance",
    "DistanceGroup": "DistanceGroup",
    
    # Aircraft
    "Tail_Number": "Tail_Number",
    "Flight_Number_Marketing_Airline": "Flight_Number_Marketing_Airline",
    "Flight_Number_Operating_Airline": "Flight_Number_Operating_Airline",
    
    # Geography
    "OriginCityName": "OriginCityName",
    "OriginState": "OriginState",
    "OriginStateName": "OriginStateName",
    "DestCityName": "DestCityName",
    "DestState": "DestState",
    "DestStateName": "DestStateName",
    
    # Time blocks
    "DepTimeBlk": "DepTimeBlk",
    "ArrTimeBlk": "ArrTimeBlk",
    
    # Calendar
    "Year": "Year",
    "Quarter": "Quarter",
    "Month": "Month",
    "DayofMonth": "DayofMonth",
    "DayOfWeek": "DayOfWeek",
}

# Seleccionar columnas: si existe mapear, si no crear con NaN
df_curated = pd.DataFrame()

for canonical, source in mapping.items():
    if source in cols:
        df_curated[canonical] = df_land[source]
    else:
        df_curated[canonical] = np.nan

print(f"✓ Columnas mapeadas: {len(df_curated.columns)}")
print(df_curated.head(3))

Columnas en landing: 61
✓ Columnas mapeadas: 39
  FlightDate            Airline Operating_Airline Origin Dest  Cancelled  \
0 2018-01-23  Endeavor Air Inc.                9E    ABY  ATL      False   
1 2018-01-24  Endeavor Air Inc.                9E    ABY  ATL      False   
2 2018-01-25  Endeavor Air Inc.                9E    ABY  ATL      False   

   Diverted  DivAirportLandings  CRSDepTime  DepTime  ...  DestCityName  \
0     False                 0.0        1202   1157.0  ...   Atlanta, GA   
1     False                 0.0        1202   1157.0  ...   Atlanta, GA   
2     False                 0.0        1202   1153.0  ...   Atlanta, GA   

   DestState  DestStateName  DepTimeBlk  ArrTimeBlk  Year  Quarter  Month  \
0         GA        Georgia   1200-1259   1300-1359  2018        1      1   
1         GA        Georgia   1200-1259   1300-1359  2018        1      1   
2         GA        Georgia   1200-1259   1300-1359  2018        1      1   

   DayofMonth  DayOfWeek  
0         

In [17]:
# Normalizar tipos de datos
print("Normalizando tipos de datos...")

# Fecha
if 'FlightDate' in df_curated.columns:
    df_curated['FlightDate'] = pd.to_datetime(df_curated['FlightDate'], errors='coerce')

# Day of month - usar dayofmonth() si está disponible, sino extraer de FlightDate
if 'DayofMonth' in df_curated.columns:
    df_curated['DayOfMonth'] = df_curated['DayofMonth'].fillna(
        df_curated['FlightDate'].dt.day
    ).astype('Int64')
    df_curated = df_curated.drop('DayofMonth', axis=1)

# Year, Quarter, Month - extraer de FlightDate si faltan
for col, method in [('Year', 'year'), ('Quarter', 'quarter'), ('Month', 'month')]:
    if col in df_curated.columns:
        df_curated[col] = df_curated[col].fillna(
            getattr(df_curated['FlightDate'].dt, method)
        ).astype('Int64')

# Numéricos
numeric_cols = ['Distance', 'AirTime', 'CRSElapsedTime', 'ActualElapsedTime',
                'DepDelayMinutes', 'ArrDelayMinutes']
for col in numeric_cols:
    if col in df_curated.columns:
        df_curated[col] = pd.to_numeric(df_curated[col], errors='coerce')

# Flags (int)
flag_cols = ['Cancelled', 'Diverted', 'DepDel15', 'ArrDel15']
for col in flag_cols:
    if col in df_curated.columns:
        df_curated[col] = pd.to_numeric(df_curated[col], errors='coerce').fillna(0).astype('int64')

# DayOfWeek como int
if 'DayOfWeek' in df_curated.columns:
    df_curated['DayOfWeek'] = pd.to_numeric(df_curated['DayOfWeek'], errors='coerce').astype('Int64')

print(f"✓ Tipos normalizados")
print(df_curated.dtypes)
print(f"\n{len(df_curated)} filas después de normalización")

Normalizando tipos de datos...
✓ Tipos normalizados
FlightDate                         datetime64[us]
Airline                                       str
Operating_Airline                             str
Origin                                        str
Dest                                          str
Cancelled                                   int64
Diverted                                    int64
DivAirportLandings                        float64
CRSDepTime                                  int64
DepTime                                   float64
CRSArrTime                                  int64
ArrTime                                   float64
DepDelayMinutes                           float64
ArrDelayMinutes                           float64
DepDel15                                    int64
ArrDel15                                    int64
DepartureDelayGroups                      float64
ArrivalDelayGroups                        float64
AirTime                                   float6

In [18]:
# Escribir curated particionado por Year y Month (simulado)
curated_path_parquet = os.path.join(PATH_CURATED, "flights_curated_parquet")
os.makedirs(curated_path_parquet, exist_ok=True)

print(f"Escribiendo curated particionado en: {curated_path_parquet}")

# Guardar cada Year/Month como subdirectorio (similar a Spark)
if 'Year' in df_curated.columns and 'Month' in df_curated.columns:
    for year in df_curated['Year'].dropna().unique():
        for month in df_curated[df_curated['Year'] == year]['Month'].dropna().unique():
            partition_path = os.path.join(
                curated_path_parquet,
                f"Year={int(year)}",
                f"Month={int(month)}"
            )
            os.makedirs(partition_path, exist_ok=True)
            
            # Filtrar datos y guardar
            mask = (df_curated['Year'] == year) & (df_curated['Month'] == month)
            df_part = df_curated[mask]
            
            part_file = os.path.join(partition_path, "data.parquet")
            df_part.to_parquet(part_file, engine='pyarrow', index=False, compression='snappy')
else:
    # Si no hay Year/Month, guardar sin particionar
    part_file = os.path.join(curated_path_parquet, "Year=unknown", "Month=unknown", "data.parquet")
    os.makedirs(os.path.dirname(part_file), exist_ok=True)
    df_curated.to_parquet(part_file, engine='pyarrow', index=False, compression='snappy')

print(f"✓ Curated escrito con {len(df_curated)} filas")

Escribiendo curated particionado en: .\curated\flights\flights_curated_parquet
✓ Curated escrito con 29193782 filas


## 4) FUNCTIONAL — Construcción de Dimensiones y Fact

In [19]:
# Función auxiliar para extraer Hora y Minuto de HHMM
def hhmm_to_hour(hhmm):
    """Extraer hora de formato HHMM"""
    if pd.isna(hhmm):
        return np.nan
    s = str(int(hhmm)).zfill(4)
    return int(s[:2])

def hhmm_to_minute(hhmm):
    """Extraer minuto de formato HHMM"""
    if pd.isna(hhmm):
        return np.nan
    s = str(int(hhmm)).zfill(4)
    return int(s[2:])

# ============================================================
# dim_fecha
# ============================================================
dim_fecha = df_curated[['FlightDate', 'Year', 'Quarter', 'Month', 'DayOfMonth', 'DayOfWeek']].copy()
dim_fecha = dim_fecha.dropna(subset=['FlightDate']).drop_duplicates(subset=['FlightDate']).sort_values('FlightDate')
dim_fecha['dim_fecha_id'] = range(1, len(dim_fecha) + 1)

# ============================================================
# dim_hora (de CRSDepTime y CRSArrTime)
# ============================================================
horas_dep = df_curated[['CRSDepTime', 'DepTimeBlk']].dropna(subset=['CRSDepTime']).copy()
horas_dep['hhmm'] = horas_dep['CRSDepTime'].astype(int)
horas_dep = horas_dep[['hhmm', 'DepTimeBlk']]

horas_arr = df_curated[['CRSArrTime', 'ArrTimeBlk']].dropna(subset=['CRSArrTime']).copy()
horas_arr['hhmm'] = horas_arr['CRSArrTime'].astype(int)
horas_arr = horas_arr[['hhmm', 'ArrTimeBlk']].rename(columns={'ArrTimeBlk': 'DepTimeBlk'})

dim_hora = pd.concat([horas_dep, horas_arr], ignore_index=True).drop_duplicates(subset=['hhmm']).sort_values('hhmm').reset_index(drop=True)
dim_hora['Hora'] = dim_hora['hhmm'].apply(hhmm_to_hour)
dim_hora['Minuto'] = dim_hora['hhmm'].apply(hhmm_to_minute)
dim_hora = dim_hora.rename(columns={'DepTimeBlk': 'TimeBlock'})
dim_hora['dim_hora_id'] = range(1, len(dim_hora) + 1)

# ============================================================
# dim_aerolinea
# ============================================================
dim_aerolinea = df_curated[['Airline']].dropna().drop_duplicates().sort_values('Airline').reset_index(drop=True)
dim_aerolinea['dim_aerolinea_id'] = range(1, len(dim_aerolinea) + 1)

# ============================================================
# dim_origen
# ============================================================
dim_origen = df_curated[['Origin', 'OriginCityName', 'OriginState', 'OriginStateName']].dropna(subset=['Origin']).drop_duplicates().sort_values('Origin').reset_index(drop=True)
dim_origen['dim_origen_id'] = range(1, len(dim_origen) + 1)

# ============================================================
# dim_destino
# ============================================================
dim_destino = df_curated[['Dest', 'DestCityName', 'DestState', 'DestStateName']].dropna(subset=['Dest']).drop_duplicates().sort_values('Dest').reset_index(drop=True)
dim_destino['dim_destino_id'] = range(1, len(dim_destino) + 1)

# ============================================================
# dim_ruta
# ============================================================
dim_ruta = df_curated[['Origin', 'Dest', 'DistanceGroup']].dropna(subset=['Origin', 'Dest']).drop_duplicates().sort_values(['Origin', 'Dest']).reset_index(drop=True)
dim_ruta['dim_ruta_id'] = range(1, len(dim_ruta) + 1)

# ============================================================
# dim_avion
# ============================================================
dim_avion = df_curated[['Tail_Number']].dropna().drop_duplicates().sort_values('Tail_Number').reset_index(drop=True)
dim_avion['dim_avion_id'] = range(1, len(dim_avion) + 1)

# ============================================================
# dim_vuelo
# ============================================================
dim_vuelo = df_curated[['Flight_Number_Marketing_Airline', 'Flight_Number_Operating_Airline']].drop_duplicates().reset_index(drop=True)
dim_vuelo['dim_vuelo_id'] = range(1, len(dim_vuelo) + 1)

print(f"dim_fecha: {len(dim_fecha)} filas")
print(f"dim_hora: {len(dim_hora)} filas")
print(f"dim_aerolinea: {len(dim_aerolinea)} filas")
print(f"dim_origen: {len(dim_origen)} filas")
print(f"dim_destino: {len(dim_destino)} filas")
print(f"dim_ruta: {len(dim_ruta)} filas")
print(f"dim_avion: {len(dim_avion)} filas")
print(f"dim_vuelo: {len(dim_vuelo)} filas")

dim_fecha: 1673 filas
dim_hora: 1440 filas
dim_aerolinea: 28 filas
dim_origen: 389 filas
dim_destino: 389 filas
dim_ruta: 8268 filas
dim_avion: 7089 filas
dim_vuelo: 9422 filas


In [20]:
# ============================================================
# Construir FACT_VUELOS mediante joins eficientes
# ============================================================

print("Construyendo fact_vuelos...")

# Usar lookups en diccionarios en lugar de merges (más eficiente en memoria)
# Crear diccionarios para lookups
dim_fecha_dict = dict(zip(dim_fecha['FlightDate'], dim_fecha['dim_fecha_id']))
dim_aerolinea_dict = dict(zip(dim_aerolinea['Airline'], dim_aerolinea['dim_aerolinea_id']))
dim_origen_dict = dict(zip(dim_origen['Origin'], dim_origen['dim_origen_id']))
dim_destino_dict = dict(zip(dim_destino['Dest'], dim_destino['dim_destino_id']))
dim_avion_dict = dict(zip(dim_avion['Tail_Number'], dim_avion['dim_avion_id']))

# Para dim_ruta (composite key)
dim_ruta_dict = {}
for _, row in dim_ruta.iterrows():
    key = (row['Origin'], row['Dest'])
    dim_ruta_dict[key] = row['dim_ruta_id']

# Para dim_vuelo (composite key)
dim_vuelo_dict = {}
for _, row in dim_vuelo.iterrows():
    key = (row['Flight_Number_Marketing_Airline'], row['Flight_Number_Operating_Airline'])
    dim_vuelo_dict[key] = row['dim_vuelo_id']

# Para dim_hora (por hhmm)
dim_hora_dict = dict(zip(dim_hora['hhmm'], dim_hora['dim_hora_id']))

print("Mappings creados. Asignando foreign keys...")

# Crear el fact table con solo las columnas necesarias (menor uso de RAM)
fact_vuelos = df_curated[[
    'FlightDate', 'Airline', 'Origin', 'Dest', 'Tail_Number',
    'Flight_Number_Marketing_Airline', 'Flight_Number_Operating_Airline',
    'CRSDepTime', 'CRSArrTime',
    'Distance', 'AirTime', 'CRSElapsedTime', 'ActualElapsedTime',
    'DepDelayMinutes', 'ArrDelayMinutes', 'DepDel15', 'ArrDel15',
    'Cancelled', 'Diverted', 'DepartureDelayGroups', 'ArrivalDelayGroups',
    'Operating_Airline', 'DepTimeBlk', 'ArrTimeBlk'
]].copy()

# Asignar foreign keys mediante lookups
fact_vuelos['dim_fecha_id'] = fact_vuelos['FlightDate'].map(dim_fecha_dict)
fact_vuelos['dim_aerolinea_id'] = fact_vuelos['Airline'].map(dim_aerolinea_dict)
fact_vuelos['dim_origen_id'] = fact_vuelos['Origin'].map(dim_origen_dict)
fact_vuelos['dim_destino_id'] = fact_vuelos['Dest'].map(dim_destino_dict)
fact_vuelos['dim_avion_id'] = fact_vuelos['Tail_Number'].map(dim_avion_dict)
fact_vuelos['dim_ruta_id'] = fact_vuelos.apply(
    lambda row: dim_ruta_dict.get((row['Origin'], row['Dest'])), axis=1
)
fact_vuelos['dim_vuelo_id'] = fact_vuelos.apply(
    lambda row: dim_vuelo_dict.get((row['Flight_Number_Marketing_Airline'], row['Flight_Number_Operating_Airline'])), axis=1
)

# Para dim_hora, convertir CRSDepTime y CRSArrTime a int
fact_vuelos['CRSDepTime_int'] = pd.to_numeric(fact_vuelos['CRSDepTime'], errors='coerce').astype('Int64')
fact_vuelos['CRSArrTime_int'] = pd.to_numeric(fact_vuelos['CRSArrTime'], errors='coerce').astype('Int64')

fact_vuelos['dim_hora_id'] = fact_vuelos['CRSDepTime_int'].map(dim_hora_dict)
fact_vuelos['dim_hora_arr_id'] = fact_vuelos['CRSArrTime_int'].map(dim_hora_dict)

# Limpiar columnas temporales
fact_vuelos = fact_vuelos.drop(['CRSDepTime_int', 'CRSArrTime_int'], axis=1)

# Crear fact_id incremental
fact_vuelos['fact_id'] = range(1, len(fact_vuelos) + 1)

# Reordenar columnas finales
fact_vuelos = fact_vuelos[[
    'fact_id',
    'dim_fecha_id', 'dim_hora_id', 'dim_hora_arr_id',
    'dim_avion_id', 'dim_vuelo_id', 'dim_origen_id', 'dim_destino_id',
    'dim_ruta_id', 'dim_aerolinea_id',
    'Distance', 'AirTime', 'CRSElapsedTime', 'ActualElapsedTime',
    'DepDelayMinutes', 'ArrDelayMinutes', 'DepDel15', 'ArrDel15',
    'Cancelled', 'Diverted', 'DepartureDelayGroups', 'ArrivalDelayGroups',
    'Operating_Airline', 'DepTimeBlk', 'ArrTimeBlk'
]]

print(f"✓ fact_vuelos: {len(fact_vuelos)} filas, {len(fact_vuelos.columns)} columnas")
print(fact_vuelos.head(3))

Construyendo fact_vuelos...
Mappings creados. Asignando foreign keys...


MemoryError: Unable to allocate 1.74 GiB for an array with shape (8, 29193782) and data type object

In [21]:
# ============================================================
# Escribir Parquet funcional
# ============================================================

path_functional_parquet = os.path.join(PATH_FUNCTIONAL, "parquet")
os.makedirs(path_functional_parquet, exist_ok=True)

print(f"Escribiendo tablas funcionales en Parquet...")

tables = {
    'dim_fecha': dim_fecha,
    'dim_hora': dim_hora,
    'dim_aerolinea': dim_aerolinea,
    'dim_origen': dim_origen,
    'dim_destino': dim_destino,
    'dim_ruta': dim_ruta,
    'dim_avion': dim_avion,
    'dim_vuelo': dim_vuelo,
    'fact_vuelos': fact_vuelos,
}

for name, df_table in tables.items():
    table_path = os.path.join(path_functional_parquet, name)
    os.makedirs(table_path, exist_ok=True)
    
    file_path = os.path.join(table_path, "data.parquet")
    df_table.to_parquet(file_path, engine='pyarrow', index=False, compression='snappy')
    
    size_mb = hr_size(os.path.getsize(file_path))
    print(f"  ✓ {name}: {size_mb}")

print(f"\n✓ Tablas funcionales escritas en: {path_functional_parquet}")

Escribiendo tablas funcionales en Parquet...
  ✓ dim_fecha: 27.80 KB
  ✓ dim_hora: 19.18 KB
  ✓ dim_aerolinea: 2.42 KB
  ✓ dim_origen: 14.21 KB
  ✓ dim_destino: 14.16 KB
  ✓ dim_ruta: 67.70 KB
  ✓ dim_avion: 83.38 KB
  ✓ dim_vuelo: 143.70 KB
  ✓ fact_vuelos: 508.97 MB

✓ Tablas funcionales escritas en: .\functional\datamart\parquet


## 5) ORC — Conversión y Comparación de Tamaños

In [ ]:
# Convertir fact_vuelos a ORC usando PyArrow
print("Convirtiendo fact_vuelos a ORC...")

try:
    import pyarrow.orc as orc
    orc_available = True
except ImportError:
    print("⚠ PyArrow ORC writer no disponible. Instalando...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow[orc]", "-q"])
    import pyarrow.orc as orc
    orc_available = True

path_functional_orc = os.path.join(PATH_FUNCTIONAL, "orc")
os.makedirs(path_functional_orc, exist_ok=True)

# Convertir solo fact_vuelos (sin convertir curated para evitar issues de timezone en Windows)
print("Escribiendo fact_vuelos en ORC...")
fact_orc_path = os.path.join(path_functional_orc, "fact_vuelos")
os.makedirs(fact_orc_path, exist_ok=True)

try:
    orc_file_fact = os.path.join(fact_orc_path, "data.orc")
    table_fact = pa.Table.from_pandas(fact_vuelos)
    orc.write_table(table_fact, orc_file_fact)
    print(f"✓ ORC escrito: {orc_file_fact}")
except Exception as e:
    print(f"⚠ Error escribiendo ORC: {str(e)}")
    print("  (Issue conocido de timezone en Windows. Usando solo Parquet.)")
    orc_file_fact = None

# Comparar tamaños: Parquet vs ORC para fact
print("\n" + "="*50)
print("COMPARACIÓN DE TAMAÑOS: PARQUET vs ORC")
print("="*50)

fact_parq_path = os.path.join(path_functional_parquet, "fact_vuelos.parquet")
if os.path.exists(fact_parq_path):
    sz_fact_parq = os.path.getsize(fact_parq_path)
    print(f"\nFact_Vuelos Parquet: {hr_size(sz_fact_parq)}")
    
    if orc_file_fact and os.path.exists(orc_file_fact):
        sz_fact_orc = os.path.getsize(orc_file_fact)
        print(f"Fact_Vuelos ORC    : {hr_size(sz_fact_orc)}")
        if sz_fact_orc > 0:
            ratio = (1 - sz_fact_orc / sz_fact_parq) * 100
            print(f"Ahorro: {ratio:.1f}%")
    else:
        print("(ORC no disponible en esta plataforma)")

Convirtiendo fact_vuelos a ORC...
Escribiendo fact_vuelos en ORC...
✓ ORC escrito: .\functional\datamart\orc\fact_vuelos\data.orc

COMPARACIÓN DE TAMAÑOS: PARQUET vs ORC


## 6) Benchmark — Comparación de Lectura y Agregación

In [ ]:
def benchmark_read_agg(read_fn, label: str, runs: int = 3):
    """Benchmark lectura + agregación"""
    times = []
    
    for run in range(runs):
        t0 = time.time()
        
        # Leer
        df_read = read_fn()
        
        # Agregación: promedio de DepDelayMinutes por dim_aerolinea_id
        result = (
            df_read
            .groupby('dim_aerolinea_id')['DepDelayMinutes']
            .mean()
            .sort_index()
        )
        
        t_elapsed = time.time() - t0
        times.append(t_elapsed)
    
    avg_time = sum(times) / len(times)
    print(f"{label:20} | Tiempos: {[f'{t:.3f}s' for t in times]} | Promedio: {avg_time:.3f}s")


print("="*70)
print("BENCHMARK: Lectura + Agregación (promedio por aerolinea)")
print("="*70 + "\n")

path_functional_parquet = os.path.join(PATH_FUNCTIONAL, "parquet")
path_functional_orc = os.path.join(PATH_FUNCTIONAL, "orc")

fact_parq_path = os.path.join(path_functional_parquet, "fact_vuelos", "data.parquet")
fact_orc_path = os.path.join(path_functional_orc, "fact_vuelos", "data.orc")

# Parquet con Pandas
if os.path.exists(fact_parq_path):
    benchmark_read_agg(
        lambda: pd.read_parquet(fact_parq_path, engine='pyarrow'),
        "Fact Parquet (Pandas)"
    )
else:
    print(f"⚠ Archivo no encontrado: {fact_parq_path}")

# ORC con PyArrow
if os.path.exists(fact_orc_path):
    benchmark_read_agg(
        lambda: orc.read_table(fact_orc_path).to_pandas(),
        "Fact ORC (PyArrow)"
    )
else:
    print(f"⚠ Archivo no encontrado: {fact_orc_path}")

print("\n✓ Benchmark completado")

BENCHMARK: Lectura + Agregación (promedio por aerolinea)

Fact Parquet (Pandas) | Tiempos: ['1.315s', '1.823s', '1.817s'] | Promedio: 1.652s
Fact ORC (PyArrow)   | Tiempos: ['5.942s', '4.898s', '5.182s'] | Promedio: 5.341s

✓ Benchmark completado


## 7) KPI Validación — OTP (On-Time Performance)

In [ ]:
# Leer fact_vuelos y calcular KPIs
print("Calculando KPI OTP...")

fact_df = pd.read_parquet(fact_parq_path, engine='pyarrow')

# Normalizar DepDel15: si es NaN tratar como 0 (no retrasado)
fact_df['DepDel15_norm'] = fact_df['DepDel15'].fillna(0).astype(int)

# Cálculos
n_flights = len(fact_df)
otp_rate = (1 - fact_df['DepDel15_norm']).mean()  # % vuelos puntuales
avg_dep_delay = fact_df['DepDelayMinutes'].mean()
pct_delayed15 = fact_df['DepDel15_norm'].mean()

print("\n" + "="*50)
print("KPI OTP (On-Time Performance)")
print("="*50)
print(f"Total vuelos              : {n_flights:,}")
print(f"OTP Rate (≤15 min delay)  : {otp_rate:.2%}")
print(f"Retraso promedio (minutos): {avg_dep_delay:.2f}")
print(f"% Vuelos retrasados >15min: {pct_delayed15:.2%}")
print("="*50)

# Mostrar como DataFrame para mejor visualización
kpi_df = pd.DataFrame({
    'Métrica': ['Total Vuelos', 'OTP Rate', 'Retraso Promedio (min)', '% Retrasados >15min'],
    'Valor': [f"{n_flights:,}", f"{otp_rate:.2%}", f"{avg_dep_delay:.2f}", f"{pct_delayed15:.2%}"]
})

print("\n")
print(kpi_df.to_string(index=False))

print("\n✓ KPI completados")

Calculando KPI OTP...

KPI OTP (On-Time Performance)
Total vuelos              : 29,193,782
OTP Rate (≤15 min delay)  : 83.17%
Retraso promedio (minutos): 12.78
% Vuelos retrasados >15min: 16.83%


               Métrica      Valor
          Total Vuelos 29,193,782
              OTP Rate     83.17%
Retraso Promedio (min)      12.78
   % Retrasados >15min     16.83%

✓ KPI completados


## Resumen — Pipeline Completo sin Spark

✓ **Ingesta (Landing)**: CSV/Parquet → Parquet local
✓ **Limpieza (Curated)**: Mapeo, normalización, tipos de datos
✓ **Dimensional (Functional)**: 8 dimensiones + 1 tabla de hechos
✓ **Formatos**: Parquet + ORC con comparativa de tamaños
✓ **Performance**: Benchmark local de lectura y agregación
✓ **KPIs**: Validación OTP (On-Time Performance)

In [22]:
df_curated[:10000].to_csv("prueba.csv", index=False)

In [ ]:
# ============================================================
# 8) MACHINE LEARNING — Clasificación de retrasos
# ============================================================

print("Iniciando sección de machine learning con datos limpios de curated...")

try:
    from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, accuracy_score
except ImportError:
    print("Instalando scikit-learn...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn", "-q"])
    from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, accuracy_score

# Usamos df_curated porque ya está limpio y contiene las columnas más útiles para modelado.
# Si queremos trabajar desde datamart, podemos usar fact_vuelos/las dimensiones en lugar de estas tablas.

ml_columns = [
    'CRSDepTime', 'CRSArrTime', 'Distance', 'DayOfWeek', 'Month',
    'Airline', 'Origin', 'Dest', 'DepTimeBlk', 'ArrTimeBlk', 'DepDel15'
]

df_ml = df_curated[ml_columns].dropna(subset=['DepDel15']).copy()

# Convertir a tipos numéricos e imputar valores simples
for col in ['CRSDepTime', 'CRSArrTime', 'Distance', 'DayOfWeek', 'Month']:
    df_ml[col] = pd.to_numeric(df_ml[col], errors='coerce')

# Filtrar filas válidas
required_numeric = ['CRSDepTime', 'CRSArrTime', 'Distance', 'DayOfWeek', 'Month']
df_ml = df_ml.dropna(subset=required_numeric)

# Reducir tamaño para entrenamiento local usando más datos si hay RAM disponible.
# Con 64 GB de RAM podemos procesar muchos más registros, por eso ampliamos el límite.
max_ml_rows = min(len(df_ml), 1000000)
df_ml = df_ml.sample(n=max_ml_rows, random_state=42)

# Features de tiempo
df_ml['DepHour'] = (df_ml['CRSDepTime'] // 100).astype('Int64')
df_ml['ArrHour'] = (df_ml['CRSArrTime'] // 100).astype('Int64')

# Top categorías para evitar demasiados dummies
for col, top_n in [('Airline', 10), ('Origin', 20), ('Dest', 20), ('DepTimeBlk', 12), ('ArrTimeBlk', 12)]:
    top_categories = df_ml[col].value_counts().nlargest(top_n).index
    df_ml[col] = df_ml[col].where(df_ml[col].isin(top_categories), 'OTHER')

# Variable objetivo
y = df_ml['DepDel15'].astype(int)

# One-hot encode variables categóricas
categorical_cols = ['Airline', 'Origin', 'Dest', 'DepTimeBlk', 'ArrTimeBlk']
X = pd.get_dummies(df_ml[
    ['Distance', 'DayOfWeek', 'Month', 'DepHour', 'ArrHour'] + categorical_cols
], columns=categorical_cols, drop_first=True)

print(f"Datos ML preparados: {len(X)} filas, {X.shape[1]} variables")
print(f"Distribución objetivo:\n{y.value_counts(normalize=True).round(4)}")

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {len(X_train)} filas, Test: {len(X_test)} filas")

# Baseline models
baseline_lr = LogisticRegression(max_iter=1000)
baseline_rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

baseline_lr.fit(X_train, y_train)
baseline_rf.fit(X_train, y_train)

print("\nBaseline completed")

# Hiperparametros para optimización
rf_param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

lr_param_grid = {
    'penalty': ['l2'],
    'C': [0.01, 0.1, 1.0, 10.0],
    'solver': ['lbfgs'],
    'class_weight': [None, 'balanced']
}

print("Iniciando optimización de hiperparámetros...")

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=20,
    scoring='accuracy',
    cv=3,
    random_state=42,
    verbose=1,
    n_jobs=-1
)

lr_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000),
    param_grid=lr_param_grid,
    scoring='accuracy',
    cv=3,
    verbose=1,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)
lr_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_
best_lr = lr_search.best_estimator_

print("\nMejores hiperparámetros Random Forest:")
print(rf_search.best_params_)
print("Best CV accuracy:", rf_search.best_score_)

print("\nMejores hiperparámetros Logistic Regression:")
print(lr_search.best_params_)
print("Best CV accuracy:", lr_search.best_score_)

for name, model in [('Baseline Logistic Regression', baseline_lr),
                    ('Mejor Logistic Regression', best_lr),
                    ('Baseline Random Forest', baseline_rf),
                    ('Mejor Random Forest', best_rf)]:
    y_pred = model.predict(X_test)
    print(f"\n--- {name} ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred, digits=4))

print("\nOptimización de hiperparámetros completada.")

Iniciando sección de machine learning con datos limpios de curated...
Datos ML preparados: 1000000 filas, 79 variables
Distribución objetivo:
DepDel15
0    0.8319
1    0.1681
Name: proportion, dtype: float64
Train: 800000 filas, Test: 200000 filas


c:\Users\fh.montalvop\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Baseline completed
Iniciando optimización de hiperparámetros...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 8 candidates, totalling 24 fits


c:\Users\fh.montalvop\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\fh.montalvop\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative s


Mejores hiperparámetros Random Forest:
{'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 30}
Best CV accuracy: 0.8319875000104124

Mejores hiperparámetros Logistic Regression:
{'C': 1.0, 'class_weight': None, 'penalty': 'l2', 'solver': 'lbfgs'}
Best CV accuracy: 0.8318624999791621

--- Baseline Logistic Regression ---
Accuracy: 0.8318
              precision    recall  f1-score   support

           0     0.8319    0.9999    0.9082    166371
           1     0.3077    0.0001    0.0002     33629

    accuracy                         0.8318    200000
   macro avg     0.5698    0.5000    0.4542    200000
weighted avg     0.7437    0.8318    0.7555    200000


--- Mejor Logistic Regression ---
Accuracy: 0.8318
              precision    recall  f1-score   support

           0     0.8319    0.9999    0.9082    166371
           1     0.3077    0.0001    0.0002     33629

    accuracy                         0.8318    200000
   macro 